# Contextual Compression with AWS

## 📖 Overview

This notebook demonstrates **Contextual Compression** for RAG using AWS services:
- **AWS OpenSearch Serverless** for document retrieval
- **Amazon Bedrock Claude** for relevance extraction
- **Amazon Bedrock Titan** for embeddings

### What is Contextual Compression?

Contextual Compression solves a common RAG problem: **retrieved documents contain irrelevant information**.

**Traditional RAG:**
1. Retrieve top-K documents (often long)
2. Pass entire documents to LLM
3. LLM must find relevant parts while ignoring noise

**Contextual Compression:**
1. Retrieve candidate documents
2. **Extract only relevant snippets** using LLM
3. Pass compressed, focused context to answer generation
4. Generate answer with cleaner context

### Why Compress Context?

**Problems with full documents:**
- Dilutes signal with noise
- Wastes token budget
- Confuses answer generation
- Increases latency and cost

**Benefits of compression:**
- Focuses on query-relevant content
- Reduces tokens (lower cost)
- Improves answer quality
- Enables more documents in context window

### When to Use

✅ **Good for:**
- Long documents (>500 words)
- When documents partially relevant
- Limited context windows
- Improving answer precision
- Reducing hallucination
- Multi-document synthesis

❌ **Not ideal for:**
- Short, focused documents
- When entire document needed
- Latency-critical apps (adds LLM call)
- Very tight budgets
- Simple keyword matching

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Generate Embedding<br/>Titan]
    B --> C[Vector Search<br/>OpenSearch]
    C --> D[Retrieve Top-10<br/>Candidate Documents]
    
    D --> E[Compression Stage<br/>For Each Document]
    E --> F[Extract Relevant Snippets<br/>Claude Haiku]
    F --> G[Filter Empty Results]
    G --> H[Compressed Context<br/>5-10 Snippets]
    
    H --> I[Generate Answer<br/>Claude Opus + Compressed Context]
    I --> J[Return Answer]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#ffe0b2
    style F fill:#fff9c4
    style G fill:#ffccbc
    style H fill:#c8e6c9
    style I fill:#b3e5fc
    style J fill:#c8e6c9
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple, Optional
import time

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'compression_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
COMPRESSOR_MODEL = 'us.anthropic.claude-haiku-3-20241022-v1:0'  # Fast for extraction
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for answers

# Compression Parameters
RETRIEVAL_TOP_K = 10  # Retrieve more candidates
COMPRESSION_TARGET = 5  # Target number of compressed snippets
MIN_SNIPPET_LENGTH = 20  # Minimum words in snippet

print(f"Configuration:")
print(f"  Initial Retrieval: Top-{RETRIEVAL_TOP_K}")
print(f"  Target Snippets: {COMPRESSION_TARGET}")
print(f"  Compressor: {COMPRESSOR_MODEL.split('.')[-1][:20]}")
print(f"  Answer LLM: {LLM_MODEL.split('.')[-1][:20]}")

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
compressor = BedrockLLM(AWS_REGION, COMPRESSOR_MODEL, temperature=0.1)  # Low temp for extraction
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Load Sample Documents

We'll use longer documents where only parts are relevant to specific queries.

In [ ]:
sample_documents = [
    # Document 1: Mixed AWS services content
    """Amazon Web Services offers a comprehensive suite of cloud services. AWS Bedrock is a fully managed service 
    that provides access to foundation models from leading AI companies through a single API. You can use models from 
    Anthropic like Claude Opus for complex reasoning tasks, Claude Sonnet for balanced performance, and Claude Haiku 
    for fast responses. The service also includes models from Amazon (Titan), Cohere, Meta, and Stability AI. Bedrock 
    provides features like fine-tuning, prompt engineering, and RAG capabilities. In addition to Bedrock, AWS offers 
    many other AI services including SageMaker for custom model training, Rekognition for image analysis, Comprehend 
    for NLP, and Polly for text-to-speech. The AWS infrastructure spans multiple regions globally.""",
    
    # Document 2: OpenSearch mixed with other services
    """AWS provides various database and search solutions. Amazon OpenSearch Service is a distributed search and 
    analytics engine based on Apache Lucene. OpenSearch Serverless is a deployment option that automatically scales 
    compute and storage resources, eliminating cluster management overhead. It supports vector search using k-NN 
    algorithms like HNSW and IVF for similarity search on embeddings. The service integrates with other AWS services 
    like Lambda, Kinesis, and CloudWatch. Other AWS database options include RDS for relational databases, DynamoDB 
    for NoSQL, Aurora for high-performance MySQL/PostgreSQL, DocumentDB for MongoDB compatibility, and Neptune for 
    graph databases. Each service has different pricing models and use cases.""",
    
    # Document 3: RAG with tangential content
    """Retrieval-Augmented Generation (RAG) is an AI framework that enhances large language models by grounding 
    responses in external knowledge sources. The typical RAG pipeline has three stages: indexing documents into a 
    vector database, retrieving relevant context based on semantic similarity, and generating responses using both 
    the query and retrieved context. This approach significantly reduces hallucinations and enables models to provide 
    accurate, up-to-date information beyond their training data. RAG systems can be implemented using various tech 
    stacks. Historical context: The RAG architecture was introduced in a 2020 paper by Facebook AI Research. Before 
    RAG, language models relied solely on parametric knowledge from training. Alternative approaches include 
    fine-tuning models on specific datasets, but this is expensive and doesn't scale well for frequently updating 
    information. RAG has become the standard for production LLM applications.""",
    
    # Document 4: Embeddings with implementation details
    """Text embeddings are numerical representations that capture semantic meaning in high-dimensional vector space. 
    Amazon Titan Embeddings V2 is AWS's latest embedding model that produces 1024-dimensional vectors optimized for 
    semantic search, clustering, and classification tasks. The model was trained on diverse text data and supports 
    English with future support for additional languages. Compared to V1, the V2 model offers improved accuracy and 
    longer context handling. Technical details: embeddings are generated through neural networks that learn to map 
    similar semantic content to nearby points in vector space. Distance metrics like cosine similarity, Euclidean 
    distance, and dot product are used to measure similarity. The dimensionality of 1024 provides a good balance 
    between expressiveness and computational efficiency. Implementation note: embeddings should be normalized for 
    cosine similarity calculations. Batch processing can improve throughput when generating embeddings for many texts.""",
    
    # Document 5: Claude models with pricing details
    """The Claude model family from Anthropic includes three variants with different capabilities and pricing. 
    Claude Opus is the most capable model, excelling at complex analysis, multi-step reasoning, mathematics, and 
    coding tasks. It's ideal for research, strategy, and advanced problem-solving but has higher latency and cost. 
    Claude Sonnet provides balanced performance suitable for most enterprise workloads including data processing, 
    analysis, and content generation. Claude Haiku is the fastest and most cost-effective option, optimized for 
    simple queries, content moderation, and high-volume tasks. Pricing varies: Opus costs approximately $15 per 
    million input tokens and $75 per million output tokens, Sonnet is $3/$15, and Haiku is $0.25/$1.25. All Claude 
    models are trained using Constitutional AI to ensure helpful, harmless, and honest responses. They support context 
    windows up to 200K tokens. The models are available through AWS Bedrock with cross-region inference profiles.""",
    
    # Document 6: Vector search algorithms
    """Vector similarity search enables finding semantically similar content by comparing embeddings in high-dimensional 
    space. The k-NN (k-nearest neighbors) algorithm finds the k most similar vectors to a query vector. Exact k-NN 
    is computationally expensive for large datasets, requiring comparison with every vector. Approximate nearest 
    neighbor (ANN) algorithms like HNSW trade small accuracy losses for dramatic speed improvements. HNSW 
    (Hierarchical Navigable Small World) builds a multi-layer graph structure where each layer contains nodes 
    representing vectors, with edges connecting similar nodes. Search starts at the top layer and navigates down, 
    progressively refining results. Key parameters include ef_construction (quality during index building) and M 
    (connections per node). OpenSearch implements HNSW through the k-NN plugin with FAISS or nmslib engines. 
    Alternative algorithms include IVF (inverted file index) which partitions the space into clusters. Performance 
    tuning requires balancing recall (finding true nearest neighbors) against latency and memory usage.""",
    
    # Document 7: RAG optimization strategies
    """Optimizing RAG systems involves multiple dimensions: retrieval quality, generation quality, latency, and cost. 
    For retrieval, strategies include using hybrid search (combining vector and keyword search), reranking results 
    with cross-encoders or LLMs, query expansion, and HyDE (generating hypothetical documents). Chunking strategy 
    significantly impacts performance - fixed-size chunking is simple but may split context, semantic chunking 
    preserves meaning boundaries, and recursive chunking handles hierarchical content. Context compression can reduce 
    tokens while maintaining relevant information. For generation, prompt engineering is crucial: clearly structure 
    the context, instruct the model to cite sources, and specify output format. Model selection matters: use Opus 
    for complex reasoning, Sonnet for balanced tasks, and Haiku for simple queries. Caching embeddings and results 
    reduces redundant computation. Monitoring is essential: track retrieval metrics (precision, recall, NDCG), 
    generation quality (faithfulness, relevance), latency, and costs. A/B testing helps validate improvements.""",
    
    # Document 8: Production considerations
    """Deploying RAG systems to production requires careful planning across infrastructure, monitoring, and operations. 
    Infrastructure considerations include choosing between managed services like OpenSearch Serverless for simplicity 
    or self-managed clusters for control. Vector indices should be sharded for large datasets, with proper replication 
    for availability. Bedrock provides automatic scaling but monitor throttling limits. Implement caching at multiple 
    levels: embedding cache for repeated queries, vector search result cache, and generation cache for common questions. 
    Rate limiting prevents cost overruns and ensures fair resource allocation. Error handling should include graceful 
    degradation when services are unavailable, with fallbacks to cached results or simpler retrieval methods. Security 
    involves encrypting data at rest and in transit, implementing proper IAM roles, and filtering sensitive content. 
    Monitoring should track request volume, error rates, latency percentiles (p50, p95, p99), token usage, and costs. 
    Set up alerts for anomalies. Logging helps debug issues and improve the system over time.""",
    
    # Document 9: Evaluation metrics
    """Evaluating RAG systems requires measuring both retrieval and generation quality. Retrieval metrics include 
    precision@k (relevant docs in top-k results), recall@k (proportion of relevant docs retrieved), Mean Reciprocal 
    Rank (MRR - position of first relevant result), and Normalized Discounted Cumulative Gain (NDCG - considers 
    relevance and ranking). For generation, assess answer quality through multiple dimensions: faithfulness (grounded 
    in retrieved context), relevance (addresses the query), completeness (covers key points), and coherence (well-structured). 
    Human evaluation remains the gold standard but is expensive. LLM-as-judge provides scalable automated evaluation 
    where a strong model like Claude Opus rates answer quality. Additional metrics include latency (time to response), 
    cost per query (API costs), and user satisfaction (thumbs up/down, engagement). Create evaluation datasets with 
    diverse queries, ground truth answers, and relevant document sets. Run evaluations regularly to catch regressions.""",
    
    # Document 10: Advanced RAG patterns
    """Beyond basic RAG, several advanced patterns improve performance for specific use cases. Multi-hop reasoning 
    requires multiple retrieval rounds, where each round's findings inform the next query. Graph RAG builds knowledge 
    graphs from documents, enabling traversal of relationships and entities. Agentic RAG gives the LLM tools to decide 
    when to retrieve, what to retrieve, and how to synthesize information across multiple sources. Self-RAG adds 
    reflection where the model critiques its own retrieval and generation steps, regenerating if quality is insufficient. 
    Corrective RAG (CRAG) includes fallback mechanisms when retrieval fails, such as web search or using parametric 
    knowledge. Adaptive RAG routes queries to different retrieval strategies based on query characteristics. These 
    patterns increase complexity and cost but handle edge cases better than simple RAG. The choice depends on requirements: 
    use simple RAG for straightforward Q&A, graph RAG for relationship queries, and agentic RAG for research tasks."""
]

print(f"Loaded {len(sample_documents)} long-form documents")
doc_lengths = [len(doc.split()) for doc in sample_documents]
print(f"Average document length: {sum(doc_lengths) / len(doc_lengths):.0f} words")
print(f"Longest document: {max(doc_lengths)} words")
print(f"Shortest document: {min(doc_lengths)} words")

## 5️⃣ Create Index and Index Documents

In [ ]:
# Create index
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

# Index documents
print("\nIndexing documents...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i, 'word_count': len(text.split())}
    })
    print(f"  Embedded document {i + 1}/{len(sample_documents)}")

indexed = opensearch.index_documents(documents)
print(f"\n✓ Indexed {indexed} documents")

## 6️⃣ Context Compression Function

Extract only query-relevant snippets from documents.

### Compression Strategy

We ask Claude to:
1. Read the full document
2. Identify parts relevant to the query
3. Extract only those relevant snippets
4. Return concise, focused content

**Key principles:**
- Keep query-relevant sentences
- Remove tangential information
- Maintain context and coherence
- Preserve factual accuracy

In [ ]:
def compress_document(document: str, query: str, min_length: int = 20) -> Optional[str]:
    """
    Extract query-relevant content from document
    
    Args:
        document: Full document text
        query: User query
        min_length: Minimum words in compressed result
    
    Returns:
        Compressed document or None if not relevant
    """
    compression_prompt = f"""
Extract ONLY the parts of the following document that are directly relevant to answering the query.

Query: {query}

Document:
{document}

Instructions:
- Extract complete sentences that help answer the query
- Remove tangential or unrelated information
- Maintain coherence and context
- If nothing is relevant, respond with only "NOT_RELEVANT"
- Do NOT add commentary, just return the relevant excerpts
- Separate multiple excerpts with a blank line

Relevant excerpts:
"""
    
    try:
        compressed = compressor.generate(compression_prompt, temperature=0.1)
        compressed = compressed.strip()
        
        # Check if marked as not relevant
        if "NOT_RELEVANT" in compressed.upper():
            return None
        
        # Check minimum length
        if len(compressed.split()) < min_length:
            return None
        
        return compressed
    
    except Exception as e:
        print(f"Error compressing document: {e}")
        return None

# Test compression
test_query = "What is AWS Bedrock?"
test_doc = sample_documents[0]  # Document about AWS services

print(f"Test Query: {test_query}\n")
print(f"Original Document ({len(test_doc.split())} words):\n{test_doc[:200]}...\n")

compressed = compress_document(test_doc, test_query, MIN_SNIPPET_LENGTH)

if compressed:
    print(f"Compressed Version ({len(compressed.split())} words):\n{compressed}\n")
    reduction = (1 - len(compressed.split()) / len(test_doc.split())) * 100
    print(f"Compression: {reduction:.1f}% reduction")
else:
    print("Document not relevant to query")

## 7️⃣ Complete Compression Pipeline

Combine retrieval, compression, and generation.

In [ ]:
def compression_rag_query(query: str,
                         retrieval_k: int = 10,
                         target_snippets: int = 5) -> Tuple[str, List[Dict], List[Dict]]:
    """
    Query using contextual compression
    
    Returns:
        (answer, compressed_docs, original_docs)
    """
    start_time = time.time()
    
    # Step 1: Retrieve candidates
    print(f"Step 1: Retrieving top-{retrieval_k} candidates...")
    query_embedding = embedder.embed_text(query)
    candidates = opensearch.vector_search(query_embedding, top_k=retrieval_k)
    retrieval_time = time.time() - start_time
    print(f"✓ Retrieved {len(candidates)} documents in {retrieval_time:.2f}s\n")
    
    # Step 2: Compress each document
    print(f"Step 2: Compressing documents...")
    compress_start = time.time()
    compressed_docs = []
    
    for i, doc in enumerate(candidates, 1):
        compressed_text = compress_document(doc['text'], query, MIN_SNIPPET_LENGTH)
        
        if compressed_text:
            compressed_docs.append({
                'text': compressed_text,
                'original_text': doc['text'],
                'metadata': doc['metadata'],
                'score': doc['score'],
                'compression_ratio': len(compressed_text.split()) / len(doc['text'].split())
            })
        
        if i % 3 == 0:
            print(f"  Compressed {i}/{len(candidates)}")
    
    compression_time = time.time() - compress_start
    print(f"✓ Compression complete in {compression_time:.2f}s")
    print(f"  Kept {len(compressed_docs)}/{len(candidates)} documents\n")
    
    # Take top snippets after compression
    final_docs = compressed_docs[:target_snippets]
    
    # Step 3: Generate answer with compressed context
    print(f"Step 3: Generating answer with {len(final_docs)} compressed snippets...")
    gen_start = time.time()
    context_texts = [doc['text'] for doc in final_docs]
    answer = llm.generate_with_context(query, context_texts)
    gen_time = time.time() - gen_start
    print(f"✓ Answer generated in {gen_time:.2f}s\n")
    
    total_time = time.time() - start_time
    print(f"Total time: {total_time:.2f}s")
    
    return answer, final_docs, candidates

# Test compression RAG
test_questions = [
    "What is AWS Bedrock and what models does it support?",
    "How does vector search work in OpenSearch?",
    "What are the key considerations for production RAG systems?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print('='*70)
    
    answer, compressed, original = compression_rag_query(
        question,
        retrieval_k=RETRIEVAL_TOP_K,
        target_snippets=COMPRESSION_TARGET
    )
    
    # Calculate token savings
    original_tokens = sum(len(doc['text'].split()) for doc in original[:COMPRESSION_TARGET])
    compressed_tokens = sum(len(doc['text'].split()) for doc in compressed)
    savings = (1 - compressed_tokens / original_tokens) * 100 if original_tokens > 0 else 0
    
    print(f"\n📊 Compression Stats:")
    print(f"  Original context: {original_tokens} words")
    print(f"  Compressed context: {compressed_tokens} words")
    print(f"  Token savings: {savings:.1f}%")
    
    print(f"\n📚 Compressed Snippets:")
    for j, doc in enumerate(compressed, 1):
        print(f"\n  {j}. [Score: {doc['score']:.4f}, Ratio: {doc['compression_ratio']:.2f}]")
        print(f"     {doc['text'][:150]}...")
    
    print(f"\n✅ Answer:")
    print(answer)

## 8️⃣ Comparison: With vs Without Compression

Compare answer quality and token usage.

In [ ]:
comparison_query = "What is RAG and how is it optimized?"

print("="*70)
print("WITH CONTEXTUAL COMPRESSION")
print("="*70)

# With compression
comp_answer, comp_docs, comp_original = compression_rag_query(
    comparison_query,
    retrieval_k=RETRIEVAL_TOP_K,
    target_snippets=COMPRESSION_TARGET
)

comp_tokens = sum(len(doc['text'].split()) for doc in comp_docs)

print("\n" + "="*70)
print("WITHOUT COMPRESSION (Traditional RAG)")
print("="*70)

# Without compression
print(f"\nRetrieving top-{COMPRESSION_TARGET} documents...")
query_emb = embedder.embed_text(comparison_query)
direct_docs = opensearch.vector_search(query_emb, top_k=COMPRESSION_TARGET)
direct_answer = llm.generate_with_context(
    comparison_query,
    [doc['text'] for doc in direct_docs]
)
print(f"✓ Retrieved {len(direct_docs)} documents\n")

direct_tokens = sum(len(doc['text'].split()) for doc in direct_docs)

# Comparison
print("="*70)
print("COMPARISON")
print("="*70)

print(f"\n📊 Token Usage:\n")
print(f"  With Compression:    {comp_tokens:>6} words")
print(f"  Without Compression: {direct_tokens:>6} words")
print(f"  Savings:             {direct_tokens - comp_tokens:>6} words ({(1 - comp_tokens/direct_tokens)*100:.1f}%)")

print(f"\n💰 Cost Impact:\n")
# Rough cost calculation (1 word ≈ 1.3 tokens, $15 per 1M input tokens for Opus)
comp_cost = (comp_tokens * 1.3 / 1_000_000) * 15
direct_cost = (direct_tokens * 1.3 / 1_000_000) * 15
print(f"  With Compression:    ${comp_cost:.6f} per query")
  print(f"  Without Compression: ${direct_cost:.6f} per query")
print(f"  Savings:             ${direct_cost - comp_cost:.6f} per query")

print(f"\n📝 Answers:\n")
print(f"With Compression ({len(comp_answer.split())} words):\n{comp_answer[:300]}...\n")
print(f"Without Compression ({len(direct_answer.split())} words):\n{direct_answer[:300]}...")

print(f"\n💡 Analysis:")
print(f"  - Compression reduces context by {(1 - comp_tokens/direct_tokens)*100:.0f}%")
print(f"  - Focuses LLM attention on relevant content")
print(f"  - Enables fitting more documents in context window")
print(f"  - Trade-off: Additional compression latency (~{1:.1f}s)")

## 9️⃣ Compression Quality Analysis

Analyze what gets kept vs removed during compression.

In [ ]:
import matplotlib.pyplot as plt

# Analyze compression for a specific query
analysis_query = "What is AWS Bedrock?"

print(f"Compression Analysis for: {analysis_query}\n")
print("="*70)

# Get documents
query_emb = embedder.embed_text(analysis_query)
docs = opensearch.vector_search(query_emb, top_k=5)

print(f"\nAnalyzing {len(docs)} documents:\n")

compression_ratios = []
labels = []

for i, doc in enumerate(docs, 1):
    original_length = len(doc['text'].split())
    compressed = compress_document(doc['text'], analysis_query, MIN_SNIPPET_LENGTH)
    
    if compressed:
        compressed_length = len(compressed.split())
        ratio = compressed_length / original_length
        compression_ratios.append(ratio)
        labels.append(f"Doc {i}")
        
        print(f"Document {i}:")
        print(f"  Original:   {original_length} words")
        print(f"  Compressed: {compressed_length} words")
        print(f"  Ratio:      {ratio:.2f} ({(1-ratio)*100:.0f}% removed)")
        print(f"  Score:      {doc['score']:.4f}")
        print(f"\n  Original preview:")
        print(f"  {doc['text'][:150]}...")
        print(f"\n  Compressed:")
        print(f"  {compressed[:150]}...\n")
    else:
        print(f"Document {i}: Filtered out (not relevant)\n")

# Visualize compression ratios
if compression_ratios:
    plt.figure(figsize=(10, 6))
    
    bars = plt.bar(labels, compression_ratios, color='skyblue', edgecolor='navy', linewidth=2)
    plt.axhline(y=1.0, color='red', linestyle='--', label='No compression', linewidth=2)
    plt.ylabel('Compression Ratio (Compressed / Original)', fontsize=12, fontweight='bold')
    plt.xlabel('Document', fontsize=12, fontweight='bold')
    plt.title(f'Contextual Compression Ratios\nQuery: "{analysis_query}"', 
             fontsize=14, fontweight='bold')
    plt.ylim(0, 1.1)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, ratio in zip(bars, compression_ratios):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{ratio:.2f}\n({(1-ratio)*100:.0f}% removed)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Summary Statistics:")
    print(f"  Average compression ratio: {sum(compression_ratios)/len(compression_ratios):.2f}")
    print(f"  Average removed: {(1 - sum(compression_ratios)/len(compression_ratios))*100:.0f}%")
    print(f"  Best compression: {min(compression_ratios):.2f} ({(1-min(compression_ratios))*100:.0f}% removed)")
    print(f"  Least compression: {max(compression_ratios):.2f} ({(1-max(compression_ratios))*100:.0f}% removed)")

## 🔟 Performance & Cost Analysis

In [ ]:
# Detailed performance breakdown
test_query = "How does semantic search work?"

print("Performance & Cost Breakdown\n")
print("="*70)

# Time each stage
t1 = time.time()
query_emb = embedder.embed_text(test_query)
embed_time = time.time() - t1

t2 = time.time()
candidates = opensearch.vector_search(query_emb, top_k=RETRIEVAL_TOP_K)
retrieval_time = time.time() - t2

t3 = time.time()
compressed_docs = []
for doc in candidates:
    compressed = compress_document(doc['text'], test_query, MIN_SNIPPET_LENGTH)
    if compressed:
        compressed_docs.append({'text': compressed})
compression_time = time.time() - t3

final_docs = compressed_docs[:COMPRESSION_TARGET]

t4 = time.time()
answer = llm.generate_with_context(test_query, [d['text'] for d in final_docs])
gen_time = time.time() - t4

total_time = embed_time + retrieval_time + compression_time + gen_time

print("⏱️  Latency Breakdown:\n")
print(f"  1. Query Embedding:        {embed_time*1000:>8.1f}ms ({embed_time/total_time*100:>5.1f}%)")
print(f"  2. Vector Retrieval:       {retrieval_time*1000:>8.1f}ms ({retrieval_time/total_time*100:>5.1f}%)")
print(f"  3. Compression ({RETRIEVAL_TOP_K} docs):   {compression_time*1000:>8.1f}ms ({compression_time/total_time*100:>5.1f}%)")
print(f"  4. Answer Generation:      {gen_time*1000:>8.1f}ms ({gen_time/total_time*100:>5.1f}%)")
print(f"  {'─'*50}")
print(f"  Total:                     {total_time*1000:>8.1f}ms")

# Calculate token/cost savings
original_tokens = sum(len(doc['text'].split()) for doc in candidates[:COMPRESSION_TARGET])
compressed_tokens = sum(len(doc['text'].split()) for doc in final_docs)
tokens_saved = original_tokens - compressed_tokens

# Cost calculation (approximate)
compression_cost = RETRIEVAL_TOP_K * 0.00025  # Haiku for compression
original_input_cost = (original_tokens * 1.3 / 1_000_000) * 15  # Opus input
compressed_input_cost = (compressed_tokens * 1.3 / 1_000_000) * 15
output_cost = 0.075  # Typical output cost

total_with_compression = 0.00002 + compression_cost + compressed_input_cost + output_cost
total_without_compression = 0.00002 + original_input_cost + output_cost

print("\n💰 Cost Breakdown (per query):\n")
print(f"  With Compression:")
print(f"    - Query embedding (Titan):     ${0.00002:.6f}")
print(f"    - Compression ({RETRIEVAL_TOP_K} × Haiku):  ${compression_cost:.6f}")
print(f"    - Input tokens (Opus):         ${compressed_input_cost:.6f}")
print(f"    - Output tokens (Opus):        ${output_cost:.5f}")
print(f"    - Total:                       ${total_with_compression:.5f}")

print(f"\n  Without Compression:")
print(f"    - Query embedding (Titan):     ${0.00002:.6f}")
print(f"    - Input tokens (Opus):         ${original_input_cost:.6f}")
print(f"    - Output tokens (Opus):        ${output_cost:.5f}")
print(f"    - Total:                       ${total_without_compression:.5f}")

net_savings = total_without_compression - total_with_compression

print(f"\n  Net Impact:                    ${net_savings:+.6f} per query")
if net_savings > 0:
    print(f"  ✓ Compression saves money despite extra LLM calls")
else:
    print(f"  ✗ Compression costs more (but improves quality)")

print(f"\n📊 Token Savings:")
print(f"  Original context:  {original_tokens} words")
print(f"  Compressed context: {compressed_tokens} words")
print(f"  Tokens saved:      {tokens_saved} words ({tokens_saved/original_tokens*100:.1f}%)")

print(f"\n🎯 Trade-off Summary:")
print(f"  Latency increase: +{compression_time:.1f}s for compression")
print(f"  Token reduction: -{tokens_saved/original_tokens*100:.0f}%")
print(f"  Cost impact: ${net_savings:+.5f} per query")
print(f"  Quality: More focused, less noise")
print(f"  Best for: Long documents, multi-doc queries")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ LLM-based context compression
✅ Relevance extraction from long documents
✅ Complete compression pipeline
✅ Comparison with uncompressed RAG
✅ Compression quality analysis
✅ Cost-benefit evaluation

### Performance Characteristics

- **Latency**: +1-2 seconds (compression overhead)
- **Token Reduction**: 40-70% typically
- **Cost**: Variable (savings on input tokens vs compression cost)
- **Quality**: Improved focus, reduced hallucination

### When to Use Contextual Compression

**Use Compression when:**
- Documents are long (>500 words)
- Documents only partially relevant
- Context window is limited
- Want to reduce hallucination
- Need to fit more documents
- Quality > speed

**Skip Compression when:**
- Documents already concise
- Entire document needed
- Latency critical
- Very tight budget
- Simple keyword matching

### Key Insights

1. **Quality vs Quantity**: Better to have 5 focused snippets than 10 noisy documents
2. **Token Economics**: Compression cost often offset by input token savings
3. **Hallucination Reduction**: Less noise = fewer hallucinations
4. **Context Window**: Compression enables fitting more logical documents
5. **Not Binary**: Can use compression selectively based on doc length

### Compression Strategies

**1. LLM-based (This Notebook)**
- Most accurate
- Understands semantics
- Higher cost
- Flexible

**2. Embedding-based**
- Score sentence relevance
- Keep high-scoring sentences
- Cheaper
- Less nuanced

**3. Keyword-based**
- Simple heuristics
- Very fast
- Lowest cost
- Misses semantics

**4. Learned Compressor**
- Fine-tune model for compression
- Best quality
- Requires training data
- Deployment overhead

### Best Practices

1. **Use Fast Model**: Haiku for compression (20x cheaper than Opus)
2. **Batch Compression**: Process multiple docs in parallel
3. **Set Minimum Length**: Filter very short compressions
4. **Cache Compressions**: For repeated queries
5. **Adaptive Threshold**: Compress only long documents

### Optimization Techniques

**Parallel Compression:**
```python
# Compress multiple documents concurrently
with ThreadPoolExecutor() as executor:
    compressed = list(executor.map(
        lambda doc: compress_document(doc, query),
        candidates
    ))
```

**Adaptive Compression:**
```python
# Only compress documents above threshold
COMPRESSION_THRESHOLD = 300  # words
if len(doc.split()) > COMPRESSION_THRESHOLD:
    compressed = compress_document(doc, query)
else:
    compressed = doc  # Keep as-is
```

**Caching:**
```python
# Cache compressed results
cache_key = f"{doc_id}:{query_hash}"
if cache_key in compression_cache:
    return compression_cache[cache_key]
```

### Limitations

1. **Added Latency**: 1-2s for compression
2. **Compression Cost**: N × Haiku calls
3. **Information Loss**: May remove useful context
4. **Over-filtering**: Might filter all documents
5. **Extraction Errors**: LLM might miss relevance

### Combining with Other Patterns

**Compression + Reranking:**
1. Retrieve 20 candidates
2. Rerank to top-10
3. Compress top-10
4. Generate answer

**Compression + Fusion:**
1. Generate query variants
2. Retrieve for each variant
3. Fuse results
4. Compress fused documents
5. Generate answer

**Hierarchical Compression:**
1. Coarse compression (paragraph level)
2. Fine compression (sentence level)
3. Two-stage approach

### Advanced Topics

- **Prompt Compression**: Compress the query too
- **Iterative Compression**: Multi-pass refinement
- **Structured Extraction**: Extract as JSON/structured data
- **Cross-document Deduplication**: Remove repeated info
- **Summarization vs Extraction**: Sometimes summarize is better

### Metrics to Track

1. **Compression Ratio**: Compressed / Original length
2. **Filter Rate**: % documents filtered out
3. **Token Savings**: Absolute and percentage
4. **Cost Impact**: Net cost change
5. **Answer Quality**: With vs without compression
6. **Latency**: Total and compression overhead

### Next Steps

- **Embedding-based compression**: Sentence-level scoring
- **Hybrid compression**: Combine LLM + embeddings
- **Dynamic thresholds**: Adapt based on doc length
- **Multi-stage compression**: Hierarchical approach
- **Fine-tuned compressor**: Train specific model

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")